# **Model Development and Offline Evaluation**

## **1. Data Preparation**

In [25]:
# Import the libraries

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score

In [2]:
# Load the dataset

wine_data = pd.read_csv("winequality-red.csv", sep=";")

wine_data.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5


In [3]:
wine_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 1599 entries, 0 to 1598
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   fixed acidity         1599 non-null   float64
 1   volatile acidity      1599 non-null   float64
 2   citric acid           1599 non-null   float64
 3   residual sugar        1599 non-null   float64
 4   chlorides             1599 non-null   float64
 5   free sulfur dioxide   1599 non-null   float64
 6   total sulfur dioxide  1599 non-null   float64
 7   density               1599 non-null   float64
 8   pH                    1599 non-null   float64
 9   sulphates             1599 non-null   float64
 10  alcohol               1599 non-null   float64
 11  quality               1599 non-null   int64  
dtypes: float64(11), int64(1)
memory usage: 150.0 KB


In [4]:
wine_data.isnull().sum()

fixed acidity           0
volatile acidity        0
citric acid             0
residual sugar          0
chlorides               0
free sulfur dioxide     0
total sulfur dioxide    0
density                 0
pH                      0
sulphates               0
alcohol                 0
quality                 0
dtype: int64

In [5]:
# Define quality class

def quality_class(score):
    if score <= 4:
        return 0
    elif score <= 6:
        return 1
    else:
        return 2

wine_data["quality_class"] = wine_data["quality"].apply(quality_class)

In [6]:
wine_data["quality_class"].value_counts().sort_index()

quality_class
0      63
1    1319
2     217
Name: count, dtype: int64

In [ ]:
# Identify features and target

X = wine_data.drop(columns=["quality", "quality_class"])
y = wine_data["quality_class"]

In [9]:
# Split data into train and test sets

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y)

print("Training:", X_train.shape)
print("Testing:", X_test.shape)

Training: (1279, 11)
Testing: (320, 11)


In [11]:
# Standardize the dataset using StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## **2. Model Training and Evaluation**

In [15]:
# Train the Logistic Regression

log_model = LogisticRegression(max_iter=1000, random_state=42)

log_model.fit(X_train_scaled, y_train)

# Predict test data
y_pred_log = log_model.predict(X_test_scaled)

In [ ]:
# Train the Random Forest

rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42)

rf_model.fit(X_train_scaled, y_train)

# Predict test data
y_pred_rf = rf_model.predict(X_test_scaled)

In [19]:
# Evaluate Accuracy and F1-score

# Logistic Regression

log_accuracy = accuracy_score(y_test, y_pred_log)
log_f1 = f1_score(y_test, y_pred_log, average="macro")

# Random Forest

rf_accuracy = accuracy_score(y_test, y_pred_rf)
rf_f1 = f1_score(y_test, y_pred_rf, average="macro")

print("Logistic Regression")
print("Accuracy:", log_accuracy)
print("F1-score:", log_f1)

print("\nRandom Forest")
print("Accuracy:", rf_accuracy)
print("F1-score:", rf_f1)

Logistic Regression
Accuracy: 0.84375
F1-score: 0.46763265911675983

Random Forest
Accuracy: 0.884375
F1-score: 0.5441194393662401


## **3. Model Comparison and Discussion**

In [20]:
# Model comparison

comparison = pd.DataFrame({
    "Model": ["Logistic Regression", "Random Forest"],
    "Accuracy": [log_accuracy, rf_accuracy],
    "F1-score": [log_f1, rf_f1]})

comparison

,Model,Accuracy,F1-score
0,Logistic Regression,0.843750,0.467633
1,Random Forest,0.884375,0.544119


### **Insight**:

Random Forest performed better than Logistic Regression, achieving a higher accuracy (88.4% vs. 84.4%) and F1-score (54.4% vs. 46.8%). Random Forest was better able to capture the complex and nonlinear relationships between the wine characteristics and quality classes. However, An accuracy of 88.4% looks strong, but an F1 of 54.4% indicates that performance is not equally strong across the three quality classes, potentially due to class imbalance.

In [30]:
import sys
!{sys.executable} -m pip install flaml


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [36]:
import flaml
print("FLAML installed successfully")

FLAML installed successfully


In [38]:
# Run AutoML

from flaml import AutoML

automl = AutoML()

automl.fit(
    X_train=X_train_scaled,
    y_train=y_train,
    task="classification",
    metric="macro_f1",
    time_budget=60,
    estimator_list=["rf", "extra_tree"])

[flaml.automl.logger: 10-03 22:44:04] {2470} INFO - task = classification
[flaml.automl.logger: 10-03 22:44:04] {2481} INFO - Evaluation method: cv
[flaml.automl.logger: 10-03 22:44:04] {2584} INFO - Minimizing error metric: 1-macro_f1
[flaml.automl.logger: 10-03 22:44:04] {2701} INFO - List of ML learners in AutoML Run: ['rf', 'extra_tree']
[flaml.automl.logger: 10-03 22:44:04] {3006} INFO - iteration 0, current learner rf
[flaml.automl.logger: 10-03 22:44:04] {3141} INFO - Estimated sufficient time budget=1816s. Estimated necessary time budget=2s.
[flaml.automl.logger: 10-03 22:44:04] {3192} INFO -  at 0.2s,	estimator rf's best error=6.9866e-01,	best estimator rf's best error=6.9866e-01
[flaml.automl.logger: 10-03 22:44:04] {3006} INFO - iteration 1, current learner extra_tree
[flaml.automl.logger: 10-03 22:44:04] {3192} INFO -  at 0.4s,	estimator extra_tree's best error=6.9866e-01,	best estimator rf's best error=6.9866e-01
[flaml.automl.logger: 10-03 22:44:04] {3006} INFO - iteratio

In [39]:
from sklearn.metrics import accuracy_score, f1_score

y_pred_automl = automl.predict(X_test_scaled)

automl_accuracy = accuracy_score(y_test, y_pred_automl)
automl_f1 = f1_score(y_test, y_pred_automl, average="macro")

print("Best AutoML model:", automl.best_estimator)
print("AutoML Accuracy:", automl_accuracy)
print("AutoML F1-score:", automl_f1)

Best AutoML model: rf
AutoML Accuracy: 0.84375
AutoML F1-score: 0.5072505521943724


### **Insights**: 

The manually trained Random Forest performed best, with 88.4% accuracy and a 54.4% F1-score, compared with Logistic Regression (84.4%, 46.8%) and FLAML AutoML (84.4%, 50.7%). Random Forest likely performed better because it can capture nonlinear relationships and interactions among wine characteristics. FLAML also selected Random Forest as its best estimator, supporting its suitability for this dataset. Overall, the manually trained Random Forest showed the best performance.